# Notebook 03 — Recurring Pattern Detection (FR4)
**Team 7 — Mansi & Samyak**

Detects weekly and monthly recurring subscriptions.
Outputs `subscription_with_patterns.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('✅ Imports ready')


## 1. Load Data

In [ ]:
DATA_PATH = r'C:\Users\Mansi\Downloads\OPUS PYTHON\AI_System_New_1\AI_System_New_1\data\raw\subscription_dataset_large.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
df = df[df['Status'] == 'SUCCESS']          # Only successful transactions
df = df[['CustomerID','Date','Description','Amount','Merchant']].copy()

print(f'Loaded {len(df):,} rows')
df.head()


## 2. Recurring Detection Logic

In [ ]:
def detect_recurring(df):
    """
    Marks each transaction as recurring (True/False) and assigns a frequency.

    Rules:
    - Group by CustomerID + Merchant
    - Need at least 3 transactions
    - Compute gaps between consecutive dates
    - Monthly: median gap 25-35 days
    - Weekly : median gap 5-10 days
    - Amount spread must be <= ₹50 (consistent pricing)
    """
    MONTHLY_RANGE = (25, 35)
    WEEKLY_RANGE  = (5, 10)
    MIN_TXNS      = 3
    AMOUNT_TOL    = 50

    result = df.copy()
    result['is_recurring'] = False
    result['frequency']    = 'one-time'

    for (cust, merchant), group in result.groupby(['CustomerID','Merchant']):
        group = group.sort_values('Date')

        if len(group) < MIN_TXNS:
            continue

        gaps = group['Date'].diff().dt.days.dropna()
        gaps = gaps[gaps > 0]

        if len(gaps) == 0:
            continue

        median_gap = gaps.median()

        if MONTHLY_RANGE[0] <= median_gap <= MONTHLY_RANGE[1]:
            freq = 'monthly'
        elif WEEKLY_RANGE[0] <= median_gap <= WEEKLY_RANGE[1]:
            freq = 'weekly'
        else:
            freq = 'irregular'

        # Only mark as recurring if amount is consistent
        amount_spread = group['Amount'].max() - group['Amount'].min()
        is_rec = freq in ('monthly','weekly') and amount_spread <= AMOUNT_TOL

        result.loc[group.index, 'is_recurring'] = is_rec
        result.loc[group.index, 'frequency']    = freq if is_rec else 'irregular'

    return result


## 3. Run & Inspect

In [ ]:
df_result = detect_recurring(df)

print('Recurring distribution:')
print(df_result['is_recurring'].value_counts())
print()
print('Frequency breakdown:')
print(df_result['frequency'].value_counts())

df_result.head()


## 4. Top Merchants by Recurring Count

In [ ]:
print('Top 10 merchants by recurring transactions:')
print(df_result.groupby('Merchant')['is_recurring'].sum().sort_values(ascending=False).head(10))


## 5. Unit Test — Sanity Check

In [ ]:
test = pd.DataFrame({
    'CustomerID': [1]*3,
    'Date':        pd.to_datetime(['2024-01-01','2024-02-01','2024-03-01']),
    'Description': ['Test']*3,
    'Amount':      [500]*3,
    'Merchant':    ['Test']*3,
})
out = detect_recurring(test)
assert out['is_recurring'].all(), 'Unit test FAILED — monthly pattern not detected'
print('✅ Unit test passed — monthly pattern correctly detected')
print(out[['CustomerID','Merchant','is_recurring','frequency']])


## 6. Save Output

In [ ]:
OUT_PATH = r'C:\Users\Samyak.Bora\Downloads\AI_SYSTEM (2)\AI_System_New_1\AI_System_New_1\data\processed\subscription_with_patterns.csv'

df_result.to_csv(OUT_PATH, index=False)
print(f'✅ Saved: {OUT_PATH}')
print(f'   Rows : {len(df_result):,}')
print(df_result[['CustomerID','Merchant','is_recurring','frequency']].head())
